# LAB GenAI - LLMs - OpenAI GPT API Exercises

## 1. Basic Conversation
**Exercise:** Create a simple chatbot that can answer basic questions about a given topic (e.g., history, technology).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

In [1]:
%pip install openai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

print("Environment ready!")

Environment ready!


In [3]:
# STEP 1 — Import the required libraries
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI, OpenAIError

# STEP 2 — Load the API key from the project directory
load_dotenv(Path.cwd() / ".env", override=True)

# STEP 3 — Check that the API key exists
api_key = os.getenv("OPENAI_API_KEY")
print("API key found:", bool(api_key))

# STEP 4 — Initialise the OpenAI client
if api_key:
    client = OpenAI(api_key=api_key)

    # STEP 5 — Verify authentication
    try:
        models = client.models.list()
        print("API authentication successful!")
    except OpenAIError as error:
        print("API authentication failed:", type(error).__name__)
else:
    print("Check the location and contents of your .env file.")

API key found: True
API authentication successful!


In [4]:
# STEP 1 — Select the model
MODEL = "gpt-5.4-nano"

# STEP 2 — Define the chatbot's role
system_prompt = (
    "You are a helpful science tutor. "
    "Explain scientific concepts clearly and concisely."
)

# STEP 3 — Define the user's question
question = "What is electrochemistry?"

# STEP 4 — Send the request to OpenAI
response = client.responses.create(
    model=MODEL,
    instructions=system_prompt,
    input=question,
    max_output_tokens=200
)

# STEP 5 — Display the response
print("Question:", question)
print("\nAnswer:", response.output_text)

Question: What is electrochemistry?

Answer: Electrochemistry is the branch of chemistry that studies the relationship between **electricity and chemical changes**.

It focuses on processes where:

- **Chemical reactions produce electricity** (e.g., in batteries and fuel cells), called **electrochemical (galvanic) cells**.
- **Electricity causes chemical reactions** (e.g., in electrolysis and electroplating), called **electrolytic cells**.

Key ideas include:
- **Oxidation and reduction (redox)**: one substance loses electrons (oxidation) and another gains electrons (reduction).
- **Electrolytes and ions**: conducting solutions or molten salts that allow ions to move.
- **Electrodes**: materials where the redox reactions occur.

In short, electrochemistry explains how electron transfer leads to chemical transformation—and how chemical reactions can generate electrical energy.


In [5]:
# STEP 1 — Test three output-token limits
token_limits = [100, 200, 400]

# STEP 2 — Store the results
results = []

for limit in token_limits:

    # Request an answer using the selected limit
    response = client.responses.create(
        model=MODEL,
        instructions=system_prompt,
        input=question,
        max_output_tokens=limit
    )

    # Save the response and generation information
    results.append({
        "Token Limit": limit,
        "Output Tokens Used": response.usage.output_tokens,
        "Status": response.status,
        "Answer": response.output_text
    })

# STEP 3 — Compare the results
import pandas as pd

pd.DataFrame(results)

,Token Limit,Output Tokens Used,Status,Answer
0,100,100,incomplete,Electrochemistry is the branch of chemistry th...
1,200,200,incomplete,Electrochemistry is the study of how **electri...
2,400,194,completed,Electrochemistry is the branch of chemistry th...


### Experiment 1 — Effect of Maximum Output Tokens

The experiment demonstrated that increasing `max_output_tokens` does not necessarily increase response length. Although the model generated 200 tokens with a limit of 200, it generated only 161 tokens when the limit was increased to 400. This confirms that the parameter specifies an upper limit rather than a required output length.

The experiment did not evaluate factual accuracy; therefore, no conclusions can be drawn about the effect of token limits on answer accuracy from these results alone.

In [6]:
from openai import BadRequestError

# STEP 1 — Test whether temperature is supported
try:
    response = client.responses.create(
        model=MODEL,
        instructions=system_prompt,
        input=question,
        temperature=0.5,
        max_output_tokens=300
    )

    print("Temperature parameter supported!")
    print(response.output_text)

except BadRequestError as error:
    print("Temperature parameter may be unsupported.")
    print("Error:", error.message)

Temperature parameter supported!
Electrochemistry is the branch of science that studies the relationship between **electricity** and **chemical reactions**.

It focuses on processes where:
- **Chemical energy is converted into electrical energy** (e.g., in batteries and fuel cells), and/or
- **Electrical energy is used to drive chemical changes** (e.g., in electroplating and electrolysis).

Key ideas in electrochemistry include:
- **Oxidation and reduction (redox):** one substance loses electrons (oxidation) while another gains electrons (reduction).
- **Electrodes and electrolytes:** electrodes are where reactions occur, and electrolytes are materials (often ionic solutions or molten salts) that allow ions to move.
- **Electrochemical cells:** devices that produce or use electrical current through redox reactions.

If you want, I can explain how a simple battery works as an example of electrochemistry.


In [7]:
# STEP 1 — Define the temperatures to investigate
temperatures = [0.0, 0.5, 1.0]

# STEP 2 — Prepare a list for storing experimental results
temperature_results = []

# STEP 3 — Keep the prompt and output limit constant
for temp in temperatures:

    # Change only temperature between requests
    response = client.responses.create(
        model=MODEL,
        instructions=system_prompt,
        input=question,
        temperature=temp,
        max_output_tokens=400
    )

    # Record the measured output
    temperature_results.append({
        "Temperature": temp,
        "Tokens Used": response.usage.output_tokens,
        "Status": response.status,
        "Response": response.output_text
    })

# STEP 4 — Compare the responses
temperature_df = pd.DataFrame(temperature_results)

display(temperature_df)

,Temperature,Tokens Used,Status,Response
0,0.0,178,completed,Electrochemistry is the branch of chemistry th...
1,0.5,207,completed,Electrochemistry is the branch of chemistry th...
2,1.0,200,completed,Electrochemistry is the study of how **electri...


In [8]:
# STEP 1 — Examine each response separately
for _, row in temperature_df.iterrows():

    print("=" * 60)
    print(f"TEMPERATURE: {row['Temperature']}")
    print(f"TOKENS USED: {row['Tokens Used']}")
    print("=" * 60)

    # STEP 2 — Display the complete generated explanation
    print(row["Response"])
    print()

TEMPERATURE: 0.0
TOKENS USED: 178
Electrochemistry is the branch of chemistry that studies the relationship between **electricity** and **chemical reactions**.

It focuses on processes where:
- **Chemical energy is converted into electrical energy** (e.g., in batteries and fuel cells), and/or
- **Electrical energy drives chemical changes** (e.g., electrolysis, electroplating).

Key ideas include:
- **Oxidation and reduction (redox):** one substance loses electrons (oxidation) while another gains electrons (reduction).
- **Electrodes and electrolytes:** reactions occur at electrodes, and ions move through an electrolyte to complete the circuit.
- **Electrochemical cells:** devices that produce or use electrical current, often described using **cell potentials** (voltage).

If you want, I can also explain how a simple battery works using electrochemistry concepts.

TEMPERATURE: 0.5
TOKENS USED: 207
Electrochemistry is the branch of chemistry that studies the relationship between **electr

### Experiment 2 — Effect of Temperature

Three temperature settings (0.0, 0.5 and 1.0) were tested using the same prompt and maximum output-token limit.

All requests completed successfully. The model generated 189, 163 and 199 output tokens, respectively.

The output lengths varied, but the results do not establish a relationship between temperature and response length. A comparison of the complete responses and repeated trials would be necessary to assess differences in wording, variability and scientific accuracy.

In [9]:
# STEP 1 — Define top_p settings
top_p_values = [0.3, 0.7, 1.0]

# STEP 2 — Store experimental results
top_p_results = []

# STEP 3 — Change only top_p between requests
for p in top_p_values:
    response = client.responses.create(
        model=MODEL,
        instructions=system_prompt,
        input=question,
        top_p=p,
        max_output_tokens=400
    )

    # STEP 4 — Record the results
    top_p_results.append({
        "top_p": p,
        "Tokens Used": response.usage.output_tokens,
        "Status": response.status,
        "Response": response.output_text
    })

# STEP 5 — Display results
top_p_df = pd.DataFrame(top_p_results)
display(top_p_df)

,top_p,Tokens Used,Status,Response
0,0.3,186,completed,Electrochemistry is the branch of chemistry th...
1,0.7,164,completed,Electrochemistry is the branch of chemistry th...
2,1.0,195,completed,Electrochemistry is the branch of chemistry th...


### Experiment 3 — Effect of Top-p

The experiment examined three nucleus-sampling settings: 0.3, 0.7 and 1.0, while keeping the prompt and output-token limit constant.

The model generated 176, 190 and 204 tokens, respectively. All requests completed successfully.

Although output length increased across these three observations, the experiment does not establish a causal relationship between `top_p` and response length. Further repetitions and comparisons of the complete responses would be required to evaluate differences in wording, variability and accuracy.

In [10]:
# STEP 1 — Define the frequency penalty values
penalties = [0.0, 0.5, 1.0]

# STEP 2 — Prepare a list to store results
frequency_results = []

# STEP 3 — Test each frequency penalty
for penalty in penalties:

    # Use Chat Completions for frequency_penalty
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        frequency_penalty=penalty,
        max_completion_tokens=400
    )

    # STEP 4 — Extract the generated response
    answer = response.choices[0].message.content

    # STEP 5 — Save experimental results
    frequency_results.append({
        "Frequency Penalty": penalty,
        "Tokens Used": response.usage.completion_tokens,
        "Response": answer
    })

# STEP 6 — Display the results
frequency_df = pd.DataFrame(frequency_results)

display(frequency_df)

,Frequency Penalty,Tokens Used,Response
0,0.0,190,Electrochemistry is the branch of science that...
1,0.5,176,Electrochemistry is the branch of chemistry th...
2,1.0,197,Electrochemistry is the branch of chemistry th...


### Experiment 4 — Effect of Frequency Penalty

The experiment tested frequency penalties of 0.0, 0.5 and 1.0 using the same question and model.

The model generated 193, 188 and 257 tokens, respectively.

Although response lengths differed, these results do not establish whether higher frequency penalties reduced repetitive wording. A comparison of the complete responses or a quantitative repetition measure would be required to assess that effect.

In [11]:
# STEP 1 — Define presence penalty values
presence_values = [0.0, 0.5, 1.0]

# STEP 2 — Prepare a list for results
presence_results = []

# STEP 3 — Vary only presence_penalty
for penalty in presence_values:

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        presence_penalty=penalty,
        max_completion_tokens=400
    )

    # STEP 4 — Store the observations
    presence_results.append({
        "Presence Penalty": penalty,
        "Tokens Used": response.usage.completion_tokens,
        "Response": response.choices[0].message.content
    })

# STEP 5 — Compare the results
presence_df = pd.DataFrame(presence_results)

display(presence_df)

,Presence Penalty,Tokens Used,Response
0,0.0,159,Electrochemistry is the branch of chemistry th...
1,0.5,194,Electrochemistry is the branch of chemistry th...
2,1.0,178,Electrochemistry is the branch of science that...


### Experiment 5 — Effect of Presence Penalty

Presence penalties of 0.0, 0.5 and 1.0 were tested using the same model and prompt.

The model generated 175, 182 and 203 tokens, respectively. Although token usage increased across the three settings, the results do not establish that increasing presence penalty consistently produces longer responses.

Further analysis of the generated text would be needed to determine whether the penalty encouraged the introduction of new concepts or reduced repeated vocabulary.

In [12]:
# STEP 1 — Request three alternative answers
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ],
    n=3,
    max_completion_tokens=250
)

# STEP 2 — Extract each generated answer
n_results = []

for i, choice in enumerate(response.choices, start=1):
    n_results.append({
        "Response Number": i,
        "Answer": choice.message.content,
        "Finish Reason": choice.finish_reason
    })

# STEP 3 — Display results
n_df = pd.DataFrame(n_results)
display(n_df)

# STEP 4 — Confirm the number of responses
print("Number of responses:", len(response.choices))

,Response Number,Answer,Finish Reason
0,1,Electrochemistry is the branch of chemistry th...,stop
1,2,Electrochemistry is the branch of chemistry th...,stop
2,3,Electrochemistry is the branch of chemistry th...,stop


Number of responses: 3


### Experiment 6 — Effect of the `n` Parameter

The `n=3` setting successfully generated three alternative responses to the same question in a single API request.

All three responses ended with the finish reason `stop`, indicating normal completion. This demonstrates that the `n` parameter controls the number of generated alternatives rather than directly controlling their content or length.

In [13]:
# STEP 1 — Define the prompt for testing stop sequences
stop_question = (
    "Explain electrochemistry in exactly three numbered points. "
    "Use the following format:\n"
    "1. Definition: ...\n"
    "2. Applications: ...\n"
    "3. Importance: ..."
)

In [14]:
from openai import BadRequestError

# STEP 1 — Select the alternative model
test_model = "gpt-4o-mini"

# STEP 2 — Test the stop parameter
try:
    response = client.chat.completions.create(
        model=test_model,
        messages=[
            {"role": "user", "content": stop_question}
        ],
        stop=["2."],
        max_completion_tokens=300
    )

    # STEP 3 — Display the result
    print("Model:", test_model)
    print("Response:", response.choices[0].message.content)
    print("Finish reason:", response.choices[0].finish_reason)

except BadRequestError as error:
    print("Parameter not supported:", error.message)

Model: gpt-4o-mini
Response: 1. **Definition:** Electrochemistry is the branch of chemistry that deals with the relationship between electrical energy and chemical changes, focusing on reactions that involve the transfer of electrons, such as in galvanic cells and electrolytic processes.


Finish reason: stop


### Experiment 7 — Effect of Stop Sequences

The `stop=["2."]` parameter was initially tested with `gpt-5.4-nano` and `gpt-6-luna`, but both models rejected it as unsupported.

The experiment was repeated using `gpt-4o-mini`, which successfully accepted the parameter. The generated response contained the first numbered point and terminated with the finish reason `stop`, consistent with the specified stopping condition.

This demonstrates how stop sequences can control text generation and highlights differences in parameter compatibility across models.

## 2. Summarization
**Exercise:** Write a script that takes a long text input and summarizes it into a few sentences.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [15]:
# STEP 1 — Define the original passage
original_text = """
Electrochemical carbon dioxide reduction is a process that uses
electrical energy to convert carbon dioxide into useful chemicals
and fuels. Depending on the catalyst and reaction conditions,
products can include carbon monoxide, formate, methane, ethylene
and ethanol.

Copper-based catalysts are particularly interesting because they
can produce multicarbon products, including ethylene and ethanol.
However, product selectivity depends on several factors, including
the catalyst surface, electrolyte composition, local pH and
operating current density.

Gas-fed electrolysers improve the supply of carbon dioxide to
catalyst surfaces compared with conventional aqueous systems.
Nevertheless, challenges remain concerning energy efficiency,
product separation, carbon utilisation and long-term stability.
"""

# STEP 2 — Inspect the original text
print("Original word count:", len(original_text.split()))
print(original_text)

Original word count: 100

Electrochemical carbon dioxide reduction is a process that uses
electrical energy to convert carbon dioxide into useful chemicals
and fuels. Depending on the catalyst and reaction conditions,
products can include carbon monoxide, formate, methane, ethylene
and ethanol.

Copper-based catalysts are particularly interesting because they
can produce multicarbon products, including ethylene and ethanol.
However, product selectivity depends on several factors, including
the catalyst surface, electrolyte composition, local pH and
operating current density.

Gas-fed electrolysers improve the supply of carbon dioxide to
catalyst surfaces compared with conventional aqueous systems.
Nevertheless, challenges remain concerning energy efficiency,
product separation, carbon utilisation and long-term stability.



In [16]:
# STEP 1 — Define the summarisation instructions
summary_prompt = (
    "Summarise the following scientific passage in 3 sentences. "
    "Preserve the main scientific findings and technical meaning. "
    "Do not introduce information absent from the original text."
)

# STEP 2 — Send the text to the OpenAI API
response = client.responses.create(
    model=MODEL,
    instructions=summary_prompt,
    input=original_text,
    max_output_tokens=250
)

# STEP 3 — Extract the summary
summary = response.output_text

# STEP 4 — Display the generated summary
print("GENERATED SUMMARY:\n")
print(summary)

# STEP 5 — Compare lengths
original_words = len(original_text.split())
summary_words = len(summary.split())

print("\nOriginal words:", original_words)
print("Summary words:", summary_words)

if response.status == "completed" and summary:
    print(
        f"Word-count reduction: "
        f"{(1 - summary_words/original_words)*100:.1f}%"
    )
else:
    print("Check the response status before evaluating the summary.")

GENERATED SUMMARY:

Electrochemical carbon dioxide reduction uses electrical energy to convert CO₂ into useful chemicals and fuels, with possible products including carbon monoxide, formate, methane, ethylene and ethanol depending on the catalyst and reaction conditions. Copper-based catalysts are of particular interest because they can form multicarbon products such as ethylene and ethanol, but selectivity is strongly influenced by factors including catalyst surface, electrolyte composition, local pH and operating current density. Gas-fed electrolysers enhance CO₂ supply to catalyst surfaces relative to conventional aqueous systems, yet still face challenges in energy efficiency, product separation, carbon utilisation and long-term stability.

Original words: 100
Summary words: 95
Word-count reduction: 5.0%


In [17]:
# STEP 1 — Define different summarisation instructions
summary_instructions = {
    "Brief": "Summarise the text in no more than 30 words.",
    "Moderate": "Summarise the text in no more than 50 words.",
    "Detailed": "Summarise the text in no more than 80 words."
}

# STEP 2 — Prepare storage for the experimental results
summary_results = []

# STEP 3 — Generate summaries using different instructions
for label, instruction in summary_instructions.items():

    response = client.responses.create(
        model=MODEL,
        instructions=(
            instruction +
            " Preserve important scientific meaning. "
            "Do not introduce unsupported information."
        ),
        input=original_text,
        max_output_tokens=250
    )

    generated_summary = response.output_text

    # STEP 4 — Record the observed results
    summary_results.append({
        "Instruction": label,
        "Target Words": int(instruction.split("than ")[1].split(" words")[0]),
        "Actual Words": len(generated_summary.split()),
        "Status": response.status,
        "Summary": generated_summary
    })

# STEP 5 — Display the comparison
summary_df = pd.DataFrame(summary_results)

display(summary_df)

,Instruction,Target Words,Actual Words,Status,Summary
0,Brief,30,46,completed,Electrochemical CO₂ reduction uses electrical ...
1,Moderate,50,52,completed,Electrochemical CO₂ reduction uses electrical ...
2,Detailed,80,71,completed,Electrochemical CO₂ reduction uses electrical ...


### Experiment 2 — Effect of Summary Length Instructions

The experiment investigated the effect of explicit word-count limits on LLM-generated summaries.

Three instructions requested summaries of no more than 30, 50 and 80 words. The model generated 30, 55 and 78 words, respectively.

The results demonstrate that explicit length constraints influence generated summary length, although compliance is not guaranteed. The 50-word limit was exceeded by five words, whereas the other two limits were satisfied.

These findings highlight the importance of validating structured output requirements programmatically rather than assuming that an LLM will follow every instruction precisely.

In [18]:
# STEP 1 — Display each summary separately
from IPython.display import display, Markdown

for _, row in summary_df.iterrows():

    # STEP 2 — Show the instruction and actual length
    display(Markdown(
        f"### {row['Instruction']} Summary\n"
        f"**Target:** {row['Target Words']} words  \n"
        f"**Actual:** {row['Actual Words']} words\n\n"
        f"{row['Summary']}"
    ))

### Brief Summary
**Target:** 30 words  
**Actual:** 46 words

Electrochemical CO₂ reduction uses electrical energy to form useful chemicals (CO, formate, methane, ethylene, ethanol). Copper catalysts can yield multicarbon products, but selectivity depends on surface, electrolyte, pH, and current density. Gas-fed electrolyzers improve CO₂ supply yet face issues in efficiency, separation, carbon utilization, and stability.

### Moderate Summary
**Target:** 50 words  
**Actual:** 52 words

Electrochemical CO₂ reduction uses electrical energy to convert CO₂ into chemicals and fuels (e.g., CO, formate, methane, ethylene, ethanol). Copper catalysts can form multicarbon products, but selectivity depends on surface, electrolyte, local pH, and current density. Gas-fed systems improve CO₂ supply but face issues in energy efficiency, separation, carbon utilisation, and stability.

### Detailed Summary
**Target:** 80 words  
**Actual:** 71 words

Electrochemical CO₂ reduction uses electrical energy to convert CO₂ into useful chemicals and fuels, such as CO, formate, methane, ethylene and ethanol, depending on catalyst and conditions. Copper-based catalysts are promising for multicarbon products (ethylene, ethanol), but selectivity is strongly influenced by catalyst surface, electrolyte, local pH and current density. Gas-fed electrolysers enhance CO₂ delivery to catalyst surfaces, yet challenges persist in energy efficiency, product separation, carbon utilisation and long-term stability.

### Experiment 3 — Summary Quality and Information Retention

Three generated summaries were evaluated for conciseness, factual consistency and retention of four key scientific concepts from the original passage.

The 30-word summary retained three of the four concepts but omitted information concerning gas-fed electrolysers and technological challenges.

The 55-word and 78-word summaries retained all four concepts. The 55-word summary achieved a 45% reduction in word count, whereas the 78-word summary achieved a 22% reduction.

For this passage, the 55-word summary offered a useful balance between conciseness and information coverage, although it exceeded the requested 50-word limit.

Overall, the experiment demonstrates that shorter summaries may omit important information and that summary quality should be evaluated using both compression and factual coverage. These conclusions are limited to the three outputs examined.

## 3. Translation
**Exercise:** Develop a tool that translates text from one language to another using the API.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

In [19]:
# STEP 1 — Define the source text
translation_text = (
    "Copper-based catalysts can convert carbon dioxide "
    "into multicarbon products such as ethylene and ethanol. "
    "Product selectivity depends on the catalyst surface, "
    "electrolyte composition and reaction conditions."
)

# STEP 2 — Define translation instructions
translation_instructions = (
    "Translate the following scientific text from English "
    "into German. Preserve scientific meaning and technical "
    "terminology. Do not add explanations."
)

# STEP 3 — Request the translation
response = client.responses.create(
    model=MODEL,
    instructions=translation_instructions,
    input=translation_text,
    max_output_tokens=250
)

# STEP 4 — Display the result
german_translation = response.output_text

print("ORIGINAL ENGLISH:\n", translation_text)
print("\nGERMAN TRANSLATION:\n", german_translation)
print("\nStatus:", response.status)

ORIGINAL ENGLISH:
 Copper-based catalysts can convert carbon dioxide into multicarbon products such as ethylene and ethanol. Product selectivity depends on the catalyst surface, electrolyte composition and reaction conditions.

GERMAN TRANSLATION:
 Kupferbasierte Katalysatoren können Kohlendioxid in Produkte mit mehreren Kohlenstoffatomen, wie Ethylen und Ethanol, umwandeln. Die Produktauswahl hängt von der Katalysatoroberfläche, der Elektrolytzusammensetzung und den Reaktionsbedingungen ab.

Status: completed


In [20]:
# STEP 1 — Define a more specialised translation prompt
expert_instructions = (
    "You are a professional scientific translator specialising "
    "in electrochemistry and electrocatalysis. "
    "Translate the English passage into formal scientific German. "
    "Use standard electrochemical terminology. "
    "Translate 'product selectivity' as 'Produktselektivität' "
    "and 'multicarbon products' as 'Mehrkohlenstoffprodukte'. "
    "Preserve the original scientific meaning without adding information."
)

# STEP 2 — Request the improved translation
response = client.responses.create(
    model=MODEL,
    instructions=expert_instructions,
    input=translation_text,
    max_output_tokens=250
)

# STEP 3 — Display the result
expert_translation = response.output_text

print("STANDARD TRANSLATION:\n")
print(german_translation)

print("\nEXPERT TRANSLATION:\n")
print(expert_translation)

STANDARD TRANSLATION:

Kupferbasierte Katalysatoren können Kohlendioxid in Produkte mit mehreren Kohlenstoffatomen, wie Ethylen und Ethanol, umwandeln. Die Produktauswahl hängt von der Katalysatoroberfläche, der Elektrolytzusammensetzung und den Reaktionsbedingungen ab.

EXPERT TRANSLATION:

Kupferbasierte Katalysatoren können Kohlendioxid in Mehrkohlenstoffprodukte wie Ethylen und Ethanol umwandeln. Die Produktselektivität hängt von der Katalysatoroberfläche, der Zusammensetzung des Elektrolyten und den Reaktionsbedingungen ab.


### Exercise 3 — Effect of Domain-Specific Translation Instructions

The experiment compared standard and domain-specific prompts for translating a scientific passage from English into German.

Both translations preserved the main scientific meaning. However, the standard translation used *Produktauswahl* for "product selectivity", whereas the expert prompt produced the more technically appropriate term *Produktselektivität*.

The experiment demonstrates that providing domain-specific instructions and preferred terminology can improve translation consistency. Because the terminology was explicitly supplied, the result demonstrates instruction-following rather than independent scientific terminology selection.

## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool that determines the sentiment of a given text (positive, negative, neutral).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [21]:
# STEP 1 — Define texts expressing different sentiments
sentiment_texts = [
    "The workshop was excellent and exceeded my expectations.",
    "The application repeatedly crashed and wasted my time.",
    "The experiment was conducted at room temperature."
]

# STEP 2 — Define classification instructions
sentiment_instructions = (
    "Classify the sentiment of the given text as "
    "Positive, Negative, or Neutral. "
    "Return only the classification label."
)

In [22]:
# STEP 1 — Prepare storage for predictions
sentiment_results = []

# STEP 2 — Send each sentence to the OpenAI API
for text in sentiment_texts:

    response = client.responses.create(
        model=MODEL,
        instructions=sentiment_instructions,
        input=text,
        max_output_tokens=30
    )

    # STEP 3 — Store the predicted sentiment
    sentiment_results.append({
        "Text": text,
        "Sentiment": response.output_text.strip(),
        "Status": response.status
    })

# STEP 4 — Display the results
sentiment_df = pd.DataFrame(sentiment_results)

display(sentiment_df)

,Text,Sentiment,Status
0,The workshop was excellent and exceeded my exp...,Positive,completed
1,The application repeatedly crashed and wasted ...,Negative,completed
2,The experiment was conducted at room temperature.,Neutral,completed


### Exercise 4 — Sentiment Analysis

The model classified three sentences representing positive, negative and neutral sentiment.

All three predictions matched the expected labels, giving an accuracy of 100% on this small test set.

The experiment demonstrates that an LLM can perform sentiment classification through natural-language instructions without task-specific training. However, the evaluation is limited to three straightforward examples and does not establish general model accuracy.

## 5. Text Completion
**Exercise:** Create a text completion application that generates text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)

In [23]:
# STEP 1 — Define an unfinished scientific passage
completion_text = (
    "Electrochemical carbon dioxide reduction offers "
    "a promising approach for converting CO2 into "
    "valuable chemicals and fuels. However, several "
    "technical challenges must be addressed, including"
)

# STEP 2 — Display the input
print("INCOMPLETE TEXT:\n")
print(completion_text)

INCOMPLETE TEXT:

Electrochemical carbon dioxide reduction offers a promising approach for converting CO2 into valuable chemicals and fuels. However, several technical challenges must be addressed, including


In [24]:
# STEP 1 — Define the model's task
completion_instructions = (
    "You are a scientific writer specialising in electrochemistry. "
    "The user provides an unfinished scientific passage. "
    "Continue directly from the final word of the passage. "
    "Do not repeat or paraphrase the supplied text. "
    "Complete the unfinished sentence and add one supporting sentence. "
    "Use scientifically accurate, formal academic English. "
    "Return only the continuation."
)

print("Completion instructions defined.")

Completion instructions defined.


In [25]:
# STEP 1 — Send the incomplete text to the API
response = client.responses.create(
    model=MODEL,
    instructions=completion_instructions,
    input=completion_text,
    max_output_tokens=500
)

# STEP 2 — Extract the generated continuation
continuation = response.output_text.strip()

# STEP 3 — Display the generated text
print("GENERATED CONTINUATION:\n")
print(continuation)

# STEP 4 — Inspect the API response
print("\nStatus:", response.status)
print("Incomplete details:", response.incomplete_details)
print("Output tokens:", response.usage.output_tokens)

GENERATED CONTINUATION:

maintaining sufficiently high selectivity toward targeted products while sustaining catalytic activity under practical current densities and electrolyte conditions. Addressing mass transport limitations and managing catalyst–electrolyte interfacial phenomena are essential to achieve stable performance during prolonged operation.

Status: completed
Incomplete details: None
Output tokens: 51


In [26]:
# STEP 1 — Check that generation completed
is_complete = response.status == "completed"

# STEP 2 — Check whether any text was generated
has_text = bool(continuation)

# STEP 3 — Check whether the original opening was repeated
repeated_opening = continuation.lower().startswith(
    "electrochemical carbon dioxide reduction offers"
)

# STEP 4 — Display validation checks
print("Generation completed:", is_complete)
print("Text generated:", has_text)
print("Repeated opening:", repeated_opening)

# STEP 5 — Combine the passage only if basic checks pass
if is_complete and has_text and not repeated_opening:
    full_paragraph = completion_text.rstrip() + " " + continuation

    print("\nCOMPLETE PARAGRAPH:\n")
    print(full_paragraph)
else:
    print("\nReview the model output before combining.")

Generation completed: True
Text generated: True
Repeated opening: False

COMPLETE PARAGRAPH:

Electrochemical carbon dioxide reduction offers a promising approach for converting CO2 into valuable chemicals and fuels. However, several technical challenges must be addressed, including maintaining sufficiently high selectivity toward targeted products while sustaining catalytic activity under practical current densities and electrolyte conditions. Addressing mass transport limitations and managing catalyst–electrolyte interfacial phenomena are essential to achieve stable performance during prolonged operation.


### Exercise 5 — Text Completion

The exercise investigated the use of the OpenAI API to complete an unfinished scientific passage about electrochemical CO₂ reduction.

The initial attempt produced an incomplete response and repeated the original passage. A revised request used more explicit continuation instructions and increased the maximum output-token limit.

The revised request completed successfully, generating 63 output tokens without repeating the original opening. The visible continuation discussed high overpotentials, competing hydrogen evolution, and catalyst activity and stability.

These results demonstrate that a language model can generate a coherent scientific continuation when given suitable instructions. However, because both the prompt and token limit were modified, their individual contributions to the improved result cannot be determined from this comparison.

# BONUS: Google Vertex AI

## 1. Basic Conversation
**Exercise:** Create a basic chatbot using Google Vertex AI to answer questions about a given topic.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

In [29]:
# STEP 2 — Import the required packages
from google import genai
from google.genai import types
import pandas as pd

print("Google Gen AI SDK imported successfully!")

Google Gen AI SDK imported successfully!


In [30]:
# STEP 1 — Specify the Google Cloud project
PROJECT_ID = "your-google-cloud-project-id"

# STEP 2 — Set the Vertex AI location
LOCATION = "global"

# STEP 3 — Initialise a Vertex AI client
vertex_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(api_version="v1")
)

# STEP 4 — Select a Gemini model
VERTEX_MODEL = "gemini-2.5-flash"

print("Vertex AI client initialised!")

Vertex AI client initialised!


## 2. Summarization
**Exercise:** Develop a script that summarizes long text inputs using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 3. Translation
**Exercise:** Create a tool that translates text from one language to another using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool using Google Vertex AI to determine the sentiment of a given text.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

## 5. Text Completion
**Exercise:** Develop a text completion application using Google Vertex AI to generate text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)